In [0]:
# Load the main CCHS PUMF CSV file
df = spark.read.csv(
    "idbfs:/2026-06-25/22/_eefad5e4-d453-4c35-bc24-8ecad45474c2",
    header=True,
    inferSchema=True
)

# Display the first few rows
display(df)

In [0]:
# List of columns to check
columns_to_check = [
    'GEOGPRV', 'DHHGAGE', 'DHH_SEX', 'GEN_01', 'GEN_05', 
    'CCC_05', 'CCC_80', 'CCC_135', 'CC1_140', 'CC1_145', 
    'CCCDGSKL', 'CCCDGCAR', 'WDMDGDIF', 'RHC_05', 'INCDGHH', 
    'HWTDGISW', 'PAADVWHO', 'SMKDVSTY', 'WTS_M'
]

# Get actual columns from the dataframe
actual_columns = set(df.columns)

# Check which columns exist and which are missing
available_columns = [col for col in columns_to_check if col in actual_columns]
missing_columns = [col for col in columns_to_check if col not in actual_columns]

# Display results
print(f"Total columns to check: {len(columns_to_check)}")
print(f"Available columns: {len(available_columns)}")
print(f"Missing columns: {len(missing_columns)}")
print()

print("=" * 50)
print("AVAILABLE COLUMNS:")
print("=" * 50)
for col in available_columns:
    print(f"  ✓ {col}")

print()
print("=" * 50)
print("MISSING COLUMNS:")
print("=" * 50)
if missing_columns:
    for col in missing_columns:
        print(f"  ✗ {col}")
else:
    print("  (None - all columns are available!)")

In [0]:
# Create new dataframe with only the 19 selected columns
project_df = df.select(
    'GEOGPRV', 'DHHGAGE', 'DHH_SEX', 'GEN_01', 'GEN_05',
    'CCC_05', 'CCC_80', 'CCC_135', 'CC1_140', 'CC1_145',
    'CCCDGSKL', 'CCCDGCAR', 'WDMDGDIF', 'RHC_05', 'INCDGHH',
    'HWTDGISW', 'PAADVWHO', 'SMKDVSTY', 'WTS_M'
)

print(f"Project DataFrame created with {len(project_df.columns)} columns and {project_df.count()} rows")
print()

# Print schema
print("Schema:")
project_df.printSchema()
print()

# Display first 10 rows
print("First 10 rows:")
display(project_df.limit(10))

In [0]:
# Create clean_df with exact column name mapping
clean_df = project_df \
    .withColumnRenamed("GEOGPRV", "province") \
    .withColumnRenamed("DHHGAGE", "age_group") \
    .withColumnRenamed("DHH_SEX", "sex") \
    .withColumnRenamed("GEN_01", "general_health") \
    .withColumnRenamed("GEN_05", "mental_health") \
    .withColumnRenamed("CCC_05", "has_diabetes") \
    .withColumnRenamed("CCC_80", "has_high_blood_pressure") \
    .withColumnRenamed("CCC_135", "has_mood_disorder") \
    .withColumnRenamed("CC1_140", "has_anxiety_disorder") \
    .withColumnRenamed("CC1_145", "has_ptsd") \
    .withColumnRenamed("CCCDGSKL", "has_musculoskeletal_condition") \
    .withColumnRenamed("CCCDGCAR", "has_cardiovascular_condition") \
    .withColumnRenamed("WDMDGDIF", "difficulty_level") \
    .withColumnRenamed("RHC_05", "has_regular_doctor") \
    .withColumnRenamed("INCDGHH", "household_income") \
    .withColumnRenamed("HWTDGISW", "weight_category") \
    .withColumnRenamed("PAADVWHO", "physical_activity") \
    .withColumnRenamed("SMKDVSTY", "smoking_status") \
    .withColumnRenamed("WTS_M", "survey_weight")

print(f"Clean DataFrame created with {len(clean_df.columns)} columns and {clean_df.count()} rows")
print()

# Print schema
print("Schema:")
clean_df.printSchema()
print()

# Display first 10 rows
print("First 10 rows:")
display(clean_df.limit(10))

# Show all column names
print("\nColumn names:")
print(clean_df.columns)

In [0]:
from pyspark.sql.functions import col, count

# Get all columns except survey_weight
categorical_columns = [c for c in clean_df.columns if c != 'survey_weight']

print(f"Analyzing {len(categorical_columns)} categorical columns...")
print("=" * 80)
print()

# Loop through each column and show unique values with counts
for column_name in sorted(categorical_columns):
    print(f"Column: {column_name}")
    print("-" * 80)
    
    # Group by the column, count occurrences, and order by value
    value_counts = clean_df.groupBy(column_name) \
        .agg(count("*").alias("count")) \
        .orderBy(column_name)
    
    # Display the results
    display(value_counts)
    print()

In [0]:
# Display all column names in clean_df
print(f"Total columns in clean_df: {len(clean_df.columns)}")
print("\nColumn names:")
print("=" * 50)
for i, col_name in enumerate(clean_df.columns, 1):
    print(f"{i:2d}. {col_name}")

In [0]:
from pyspark.sql.functions import when, col

# Create labelled_df from clean_df with added label columns
labelled_df = clean_df

# Add general_health_label column
labelled_df = labelled_df.withColumn(
    "general_health_label",
    when(col("general_health") == 1, "Excellent")
    .when(col("general_health") == 2, "Very good")
    .when(col("general_health") == 3, "Good")
    .when(col("general_health") == 4, "Fair")
    .when(col("general_health") == 5, "Poor")
    .when(col("general_health") == 9, "Not stated")
    .otherwise("Unknown")
)

# Add mental_health_label column
labelled_df = labelled_df.withColumn(
    "mental_health_label",
    when(col("mental_health") == 1, "Excellent")
    .when(col("mental_health") == 2, "Very good")
    .when(col("mental_health") == 3, "Good")
    .when(col("mental_health") == 4, "Fair")
    .when(col("mental_health") == 5, "Poor")
    .when(col("mental_health") == 9, "Not stated")
    .otherwise("Unknown")
)

# Add age_group_label column
labelled_df = labelled_df.withColumn(
    "age_group_label",
    when(col("age_group") == 1, "12 to 17 years")
    .when(col("age_group") == 2, "18 to 34 years")
    .when(col("age_group") == 3, "35 to 49 years")
    .when(col("age_group") == 4, "50 to 64 years")
    .when(col("age_group") == 5, "65 and older")
    .otherwise("Unknown")
)

print(f"Labelled DataFrame created with {len(labelled_df.columns)} columns")
print(f"Added 3 new label columns: general_health_label, mental_health_label, age_group_label")
print()

# Display the requested columns
print("Showing coded columns alongside their labels:")
display(
    labelled_df.select(
        "general_health", "general_health_label",
        "mental_health", "mental_health_label",
        "age_group", "age_group_label"
    ).limit(20)
)

In [0]:
from pyspark.sql.functions import when, col

# Create labelled_df_2 from labelled_df with added label columns
labelled_df_2 = labelled_df

# Add sex_label column
labelled_df_2 = labelled_df_2.withColumn(
    "sex_label",
    when(col("sex") == 1, "Male")
    .when(col("sex") == 2, "Female")
    .otherwise("Unknown")
)

# Add province_label column
labelled_df_2 = labelled_df_2.withColumn(
    "province_label",
    when(col("province") == 10, "Newfoundland and Labrador")
    .when(col("province") == 11, "Prince Edward Island")
    .when(col("province") == 12, "Nova Scotia")
    .when(col("province") == 13, "New Brunswick")
    .when(col("province") == 24, "Quebec")
    .when(col("province") == 35, "Ontario")
    .when(col("province") == 46, "Manitoba")
    .when(col("province") == 47, "Saskatchewan")
    .when(col("province") == 48, "Alberta")
    .when(col("province") == 59, "British Columbia")
    .when(col("province") == 60, "Yukon / Northwest Territories / Nunavut")
    .otherwise("Unknown")
)

# Add has_regular_doctor_label column (OFFICIAL CCHS 2022 MAPPING)
labelled_df_2 = labelled_df_2.withColumn(
    "has_regular_doctor_label",
    when(col("has_regular_doctor") == 1, "Family doctor or general practitioner")
    .when(col("has_regular_doctor") == 2, "Medical specialist")
    .when(col("has_regular_doctor") == 3, "Nurse practitioner")
    .when(col("has_regular_doctor") == 4, "Other")
    .when(col("has_regular_doctor") == 5, "Don't have a regular health care provider")
    .when(col("has_regular_doctor") == 9, "Not stated")
    .otherwise("Unknown")
)

# Add difficulty_level_label column (CORRECTED MAPPING)
labelled_df_2 = labelled_df_2.withColumn(
    "difficulty_level_label",
    when(col("difficulty_level") == 1, "No difficulty in any domain")
    .when(col("difficulty_level") == 2, "At least some difficulty in at least one domain")
    .when(col("difficulty_level") == 6, "Valid skip")
    .when(col("difficulty_level") == 9, "Not stated")
    .otherwise("Unknown")
)

print(f"Labelled DataFrame 2 created with {len(labelled_df_2.columns)} columns")
print(f"Added 4 new label columns: sex_label, province_label, has_regular_doctor_label, difficulty_level_label")
print()

# Display the requested columns
print("Showing coded columns alongside their labels:")
display(
    labelled_df_2.select(
        "sex", "sex_label",
        "province", "province_label",
        "has_regular_doctor", "has_regular_doctor_label",
        "difficulty_level", "difficulty_level_label"
    ).limit(20)
)

In [0]:
from pyspark.sql.functions import col, count

print("VALUE DISTRIBUTIONS FOR CONFIRMATION")
print("=" * 80)
print()
print("Please review these value distributions before mapping labels:")
print()

# Columns to show value distributions for
columns_to_check = ['household_income', 'weight_category', 'physical_activity', 'smoking_status']

for column_name in columns_to_check:
    print(f"Column: {column_name}")
    print("-" * 80)
    
    # Group by the column, count occurrences, and order by value
    value_counts = labelled_df_2.groupBy(column_name) \
        .agg(count("*").alias("count")) \
        .orderBy(column_name)
    
    # Display the results
    display(value_counts)
    print()

In [0]:
from pyspark.sql.functions import when, col

# Create labelled_df_3 from labelled_df_2 with added label columns for chronic conditions
labelled_df_3 = labelled_df_2

# Add label columns for chronic condition variables
# All use the same mapping: 1=Yes, 2=No, 6=Valid skip, 9=Not stated

chronic_conditions = [
    ('has_diabetes', 'has_diabetes_label'),
    ('has_high_blood_pressure', 'has_high_blood_pressure_label'),
    ('has_mood_disorder', 'has_mood_disorder_label'),
    ('has_anxiety_disorder', 'has_anxiety_disorder_label'),
    ('has_ptsd', 'has_ptsd_label'),
    ('has_musculoskeletal_condition', 'has_musculoskeletal_condition_label'),
    ('has_cardiovascular_condition', 'has_cardiovascular_condition_label')
]

for original_col, label_col in chronic_conditions:
    labelled_df_3 = labelled_df_3.withColumn(
        label_col,
        when(col(original_col) == 1, "Yes")
        .when(col(original_col) == 2, "No")
        .when(col(original_col) == 6, "Valid skip")
        .when(col(original_col) == 9, "Not stated")
        .otherwise("Unknown")
    )

print(f"Labelled DataFrame 3 created with {len(labelled_df_3.columns)} columns")
print(f"Added 7 new label columns for chronic conditions")
print()

# Display the chronic condition columns with their labels
print("Chronic Condition Columns with Labels (first 20 rows):")
print()
display(
    labelled_df_3.select(
        "has_diabetes", "has_diabetes_label",
        "has_high_blood_pressure", "has_high_blood_pressure_label",
        "has_mood_disorder", "has_mood_disorder_label",
        "has_anxiety_disorder", "has_anxiety_disorder_label",
        "has_ptsd", "has_ptsd_label",
        "has_musculoskeletal_condition", "has_musculoskeletal_condition_label",
        "has_cardiovascular_condition", "has_cardiovascular_condition_label"
    ).limit(20)
)

In [0]:
from pyspark.sql.functions import when, col

# Create labelled_df_4 from labelled_df_3 with added label columns
labelled_df_4 = labelled_df_3

# Add household_income_label column
labelled_df_4 = labelled_df_4.withColumn(
    "household_income_label",
    when(col("household_income") == 1, "Less than $20,000")
    .when(col("household_income") == 2, "$20,000 to $39,999")
    .when(col("household_income") == 3, "$40,000 to $59,999")
    .when(col("household_income") == 4, "$60,000 to $79,999")
    .when(col("household_income") == 5, "$80,000 or more")
    .when(col("household_income") == 9, "Not stated")
    .otherwise("Unknown")
)

# Add weight_category_label column
labelled_df_4 = labelled_df_4.withColumn(
    "weight_category_label",
    when(col("weight_category") == 1, "Underweight or normal weight")
    .when(col("weight_category") == 2, "Overweight or obese")
    .when(col("weight_category") == 6, "Valid skip")
    .when(col("weight_category") == 9, "Not stated")
    .otherwise("Unknown")
)

print(f"Labelled DataFrame 4 created with {len(labelled_df_4.columns)} columns")
print(f"Added 2 new label columns: household_income_label, weight_category_label")
print()

# Display the requested columns
print("Showing coded columns alongside their labels:")
display(
    labelled_df_4.select(
        "household_income", "household_income_label",
        "weight_category", "weight_category_label"
    ).limit(20)
)

In [0]:
from pyspark.sql.functions import when, col

# Create analysis_df from labelled_df_4
analysis_df = labelled_df_4

# 1. Add poor_general_health_flag
analysis_df = analysis_df.withColumn(
    "poor_general_health_flag",
    when(col("general_health").isin([4, 5]), 1).otherwise(0)
)

# 2. Add poor_mental_health_flag
analysis_df = analysis_df.withColumn(
    "poor_mental_health_flag",
    when(col("mental_health").isin([4, 5]), 1).otherwise(0)
)

# 3. Add has_difficulty_flag (CORRECTED: 2 = has difficulty)
analysis_df = analysis_df.withColumn(
    "has_difficulty_flag",
    when(col("difficulty_level") == 2, 1).otherwise(0)
)

# 4. Add no_regular_doctor_flag (CORRECTED: only 5 = no regular provider)
analysis_df = analysis_df.withColumn(
    "no_regular_doctor_flag",
    when(col("has_regular_doctor") == 5, 1).otherwise(0)
)

# 5. Add chronic_condition_count
# Count how many chronic conditions equal 1
analysis_df = analysis_df.withColumn(
    "chronic_condition_count",
    (when(col("has_diabetes") == 1, 1).otherwise(0) +
     when(col("has_high_blood_pressure") == 1, 1).otherwise(0) +
     when(col("has_mood_disorder") == 1, 1).otherwise(0) +
     when(col("has_anxiety_disorder") == 1, 1).otherwise(0) +
     when(col("has_ptsd") == 1, 1).otherwise(0) +
     when(col("has_musculoskeletal_condition") == 1, 1).otherwise(0) +
     when(col("has_cardiovascular_condition") == 1, 1).otherwise(0))
)

# 6. Add high_vulnerability_flag
# Count how many vulnerability indicators are true
analysis_df = analysis_df.withColumn(
    "vulnerability_count",
    (col("poor_general_health_flag") +
     col("poor_mental_health_flag") +
     col("has_difficulty_flag") +
     col("no_regular_doctor_flag") +
     when(col("chronic_condition_count") >= 2, 1).otherwise(0))
)

analysis_df = analysis_df.withColumn(
    "high_vulnerability_flag",
    when(col("vulnerability_count") >= 2, 1).otherwise(0)
)

# Drop the temporary vulnerability_count column
analysis_df = analysis_df.drop("vulnerability_count")

print(f"Analysis DataFrame created with {len(analysis_df.columns)} columns")
print(f"Added 6 new flag columns")
print()

# Display the new flag columns
print("New Flag Columns (first 20 rows):")
print()
display(
    analysis_df.select(
        "poor_general_health_flag",
        "poor_mental_health_flag",
        "has_difficulty_flag",
        "no_regular_doctor_flag",
        "chronic_condition_count",
        "high_vulnerability_flag"
    ).limit(20)
)

In [0]:
from pyspark.sql.functions import col, count, sum as spark_sum, lit, round as spark_round
import pandas as pd

# Calculate summary statistics
total_count = analysis_df.count()

vulnerable_count = analysis_df.filter(col("high_vulnerability_flag") == 1).count()

not_vulnerable_count = analysis_df.filter(col("high_vulnerability_flag") == 0).count()

percentage_vulnerable = (vulnerable_count / total_count) * 100

# Calculate weighted percentage using survey_weight
total_weighted_population = analysis_df.select(spark_sum("survey_weight")).collect()[0][0]

vulnerable_weighted_population = analysis_df \
    .filter(col("high_vulnerability_flag") == 1) \
    .select(spark_sum("survey_weight")) \
    .collect()[0][0]

weighted_percentage_vulnerable = (vulnerable_weighted_population / total_weighted_population) * 100

# Create summary table as a pandas dataframe for display
summary_data = {
    "Metric": [
        "Total Respondents",
        "High Vulnerability (Count)",
        "Not Vulnerable (Count)",
        "High Vulnerability (%)",
        "High Vulnerability (Weighted %)"
    ],
    "Value": [
        f"{total_count:,}",
        f"{vulnerable_count:,}",
        f"{not_vulnerable_count:,}",
        f"{percentage_vulnerable:.2f}%",
        f"{weighted_percentage_vulnerable:.2f}%"
    ]
}

summary_df = pd.DataFrame(summary_data)

print("Vulnerability Analysis Summary")
print("=" * 80)
print()
display(summary_df)

In [0]:
from pyspark.sql.functions import col, count, sum as spark_sum, when, round as spark_round

# Group by age_group and age_group_label
age_vulnerability_analysis = analysis_df.groupBy("age_group", "age_group_label").agg(
    # Total respondents in each age group
    count("*").alias("total_respondents"),
    
    # Count of high vulnerability cases (where flag = 1)
    spark_sum(when(col("high_vulnerability_flag") == 1, 1).otherwise(0)).alias("high_vulnerability_count"),
    
    # Sum of survey weights for total population
    spark_sum("survey_weight").alias("total_weighted_population"),
    
    # Sum of survey weights for vulnerable population
    spark_sum(when(col("high_vulnerability_flag") == 1, col("survey_weight")).otherwise(0)).alias("vulnerable_weighted_population")
)

# Calculate percentages
age_vulnerability_analysis = age_vulnerability_analysis.withColumn(
    "high_vulnerability_percentage",
    spark_round((col("high_vulnerability_count") / col("total_respondents")) * 100, 2)
)

age_vulnerability_analysis = age_vulnerability_analysis.withColumn(
    "weighted_high_vulnerability_percentage",
    spark_round((col("vulnerable_weighted_population") / col("total_weighted_population")) * 100, 2)
)

# Select and order the final columns
result = age_vulnerability_analysis.select(
    "age_group",
    "age_group_label",
    "total_respondents",
    "high_vulnerability_count",
    "high_vulnerability_percentage",
    "weighted_high_vulnerability_percentage"
).orderBy("age_group")

print("Vulnerability Analysis by Age Group")
print("=" * 80)
print()
display(result)

In [0]:
from pyspark.sql.functions import col, count, sum as spark_sum, avg, round as spark_round

# Group by age_group and age_group_label and calculate component metrics
age_components_analysis = analysis_df.groupBy("age_group", "age_group_label").agg(
    # Total respondents
    count("*").alias("total_respondents"),
    
    # Count each vulnerability component
    spark_sum("poor_general_health_flag").alias("poor_general_health_count"),
    spark_sum("poor_mental_health_flag").alias("poor_mental_health_count"),
    spark_sum("has_difficulty_flag").alias("has_difficulty_count"),
    spark_sum("no_regular_doctor_flag").alias("no_regular_doctor_count"),
    
    # Average chronic condition count
    avg("chronic_condition_count").alias("avg_chronic_conditions"),
    
    # High vulnerability count
    spark_sum("high_vulnerability_flag").alias("high_vulnerability_count")
)

# Calculate percentages for each component
age_components_analysis = age_components_analysis.withColumn(
    "poor_general_health_pct",
    spark_round((col("poor_general_health_count") / col("total_respondents")) * 100, 2)
)

age_components_analysis = age_components_analysis.withColumn(
    "poor_mental_health_pct",
    spark_round((col("poor_mental_health_count") / col("total_respondents")) * 100, 2)
)

age_components_analysis = age_components_analysis.withColumn(
    "difficulty_pct",
    spark_round((col("has_difficulty_count") / col("total_respondents")) * 100, 2)
)

age_components_analysis = age_components_analysis.withColumn(
    "no_regular_doctor_pct",
    spark_round((col("no_regular_doctor_count") / col("total_respondents")) * 100, 2)
)

age_components_analysis = age_components_analysis.withColumn(
    "avg_chronic_conditions",
    spark_round(col("avg_chronic_conditions"), 2)
)

age_components_analysis = age_components_analysis.withColumn(
    "high_vulnerability_pct",
    spark_round((col("high_vulnerability_count") / col("total_respondents")) * 100, 2)
)

# Select final columns for display
result = age_components_analysis.select(
    "age_group",
    "age_group_label",
    "total_respondents",
    "poor_general_health_pct",
    "poor_mental_health_pct",
    "difficulty_pct",
    "no_regular_doctor_pct",
    "avg_chronic_conditions",
    "high_vulnerability_pct"
).orderBy("age_group")

print("Vulnerability Components Analysis by Age Group")
print("=" * 80)
print()
display(result)

In [0]:
from pyspark.sql.functions import col, count

print("DIFFICULTY_LEVEL VARIABLE INFORMATION")
print("=" * 80)
print()

# 1. Show original column name mapping
print("1. ORIGINAL COLUMN NAME MAPPING")
print("-" * 80)
print("Original CCHS Variable Name: WDMDGDIF")
print("Renamed Column Name: difficulty_level")
print()
print("Description: This variable captures whether respondents have difficulty")
print("due to a mood disorder (from the Canadian Community Health Survey).")
print()

# 2. Show value distribution for difficulty_level (coded values)
print("2. VALUE DISTRIBUTION FOR 'difficulty_level' (Coded Values)")
print("-" * 80)
difficulty_values = analysis_df.groupBy("difficulty_level") \
    .agg(count("*").alias("count")) \
    .orderBy("difficulty_level")

print("\nCoded values:")
display(difficulty_values)
print()

# 3. Show value distribution for difficulty_level_label (Human-readable labels)
print("3. VALUE DISTRIBUTION FOR 'difficulty_level_label' (Human-Readable Labels)")
print("-" * 80)
difficulty_labels = analysis_df.groupBy("difficulty_level", "difficulty_level_label") \
    .agg(count("*").alias("count")) \
    .orderBy("difficulty_level")

print("\nLabelled values:")
display(difficulty_labels)
print()

# 4. Data dictionary / codebook information
print("4. DATA DICTIONARY INFORMATION FOR WDMDGDIF")
print("-" * 80)
print("Variable: WDMDGDIF (Difficulty due to mood disorder)")
print()
print("Code Mapping:")
print("  1 = No difficulty in any domain")
print("  2 = At least some difficulty in at least one domain")
print("  6 = Valid skip (question not applicable/asked)")
print("  9 = Not stated (refused/don't know/missing)")
print()
print("Source: Canadian Community Health Survey (CCHS) - Annual Component")
print("Context: This variable measures functional difficulty or limitations")
print("         experienced by individuals with mood disorders.")
print()
print("Note: This is a derived variable that indicates whether having a mood")
print("      disorder causes difficulty in daily activities or functioning.")

In [0]:
from pyspark.sql.functions import col, sum as spark_sum
import pandas as pd

# Calculate overall vulnerability statistics
total_count = analysis_df.count()

vulnerable_count = analysis_df.filter(col("high_vulnerability_flag") == 1).count()

not_vulnerable_count = analysis_df.filter(col("high_vulnerability_flag") == 0).count()

percentage_vulnerable = (vulnerable_count / total_count) * 100

# Calculate weighted percentage using survey_weight
total_weighted_population = analysis_df.select(spark_sum("survey_weight")).collect()[0][0]

vulnerable_weighted_population = analysis_df \
    .filter(col("high_vulnerability_flag") == 1) \
    .select(spark_sum("survey_weight")) \
    .collect()[0][0]

weighted_percentage_vulnerable = (vulnerable_weighted_population / total_weighted_population) * 100

# Create summary table as a pandas dataframe for display
summary_data = {
    "Metric": [
        "Total Respondents",
        "High Vulnerability (Count)",
        "Not Vulnerable (Count)",
        "High Vulnerability (%)",
        "High Vulnerability (Weighted %)"
    ],
    "Value": [
        f"{total_count:,}",
        f"{vulnerable_count:,}",
        f"{not_vulnerable_count:,}",
        f"{percentage_vulnerable:.2f}%",
        f"{weighted_percentage_vulnerable:.2f}%"
    ]
}

summary_df = pd.DataFrame(summary_data)

print("HIGH VULNERABILITY SUMMARY (CORRECTED MAPPING)")
print("=" * 80)
print()
display(summary_df)

In [0]:
from pyspark.sql.functions import col, count, sum as spark_sum, avg, when, round as spark_round

# Group by household_income and household_income_label
income_vulnerability_analysis = analysis_df.groupBy("household_income", "household_income_label").agg(
    # Total respondents in each income group
    count("*").alias("total_respondents"),
    
    # Count of high vulnerability cases
    spark_sum(when(col("high_vulnerability_flag") == 1, 1).otherwise(0)).alias("high_vulnerability_count"),
    
    # Sum of survey weights for total and vulnerable populations
    spark_sum("survey_weight").alias("total_weighted_population"),
    spark_sum(when(col("high_vulnerability_flag") == 1, col("survey_weight")).otherwise(0)).alias("vulnerable_weighted_population"),
    
    # Count each vulnerability component
    spark_sum("poor_general_health_flag").alias("poor_general_health_count"),
    spark_sum("poor_mental_health_flag").alias("poor_mental_health_count"),
    spark_sum("no_regular_doctor_flag").alias("no_regular_doctor_count"),
    
    # Average chronic condition count
    avg("chronic_condition_count").alias("avg_chronic_conditions")
)

# Calculate high vulnerability percentage
income_vulnerability_analysis = income_vulnerability_analysis.withColumn(
    "high_vulnerability_percentage",
    spark_round((col("high_vulnerability_count") / col("total_respondents")) * 100, 2)
)

# Calculate weighted high vulnerability percentage
income_vulnerability_analysis = income_vulnerability_analysis.withColumn(
    "weighted_high_vulnerability_percentage",
    spark_round((col("vulnerable_weighted_population") / col("total_weighted_population")) * 100, 2)
)

# Calculate component percentages
income_vulnerability_analysis = income_vulnerability_analysis.withColumn(
    "poor_general_health_pct",
    spark_round((col("poor_general_health_count") / col("total_respondents")) * 100, 2)
)

income_vulnerability_analysis = income_vulnerability_analysis.withColumn(
    "poor_mental_health_pct",
    spark_round((col("poor_mental_health_count") / col("total_respondents")) * 100, 2)
)

income_vulnerability_analysis = income_vulnerability_analysis.withColumn(
    "no_regular_doctor_pct",
    spark_round((col("no_regular_doctor_count") / col("total_respondents")) * 100, 2)
)

# Round average chronic conditions
income_vulnerability_analysis = income_vulnerability_analysis.withColumn(
    "avg_chronic_conditions",
    spark_round(col("avg_chronic_conditions"), 2)
)

# Select final columns for display
result = income_vulnerability_analysis.select(
    "household_income",
    "household_income_label",
    "total_respondents",
    "high_vulnerability_count",
    "high_vulnerability_percentage",
    "weighted_high_vulnerability_percentage",
    "poor_general_health_pct",
    "poor_mental_health_pct",
    "no_regular_doctor_pct",
    "avg_chronic_conditions"
).orderBy("household_income")

print("Vulnerability Analysis by Household Income")
print("=" * 80)
print()
display(result)

In [0]:
from pyspark.sql.functions import col, count, sum as spark_sum, avg, when, round as spark_round

# Group by province and province_label
province_vulnerability_analysis = analysis_df.groupBy("province", "province_label").agg(
    # Total respondents in each province
    count("*").alias("total_respondents"),
    
    # Count of high vulnerability cases
    spark_sum(when(col("high_vulnerability_flag") == 1, 1).otherwise(0)).alias("high_vulnerability_count"),
    
    # Sum of survey weights for total and vulnerable populations
    spark_sum("survey_weight").alias("total_weighted_population"),
    spark_sum(when(col("high_vulnerability_flag") == 1, col("survey_weight")).otherwise(0)).alias("vulnerable_weighted_population"),
    
    # Count each vulnerability component
    spark_sum("poor_general_health_flag").alias("poor_general_health_count"),
    spark_sum("poor_mental_health_flag").alias("poor_mental_health_count"),
    spark_sum("no_regular_doctor_flag").alias("no_regular_doctor_count"),
    
    # Average chronic condition count
    avg("chronic_condition_count").alias("avg_chronic_conditions")
)

# Calculate high vulnerability percentage
province_vulnerability_analysis = province_vulnerability_analysis.withColumn(
    "high_vulnerability_percentage",
    spark_round((col("high_vulnerability_count") / col("total_respondents")) * 100, 2)
)

# Calculate weighted high vulnerability percentage
province_vulnerability_analysis = province_vulnerability_analysis.withColumn(
    "weighted_high_vulnerability_percentage",
    spark_round((col("vulnerable_weighted_population") / col("total_weighted_population")) * 100, 2)
)

# Calculate component percentages
province_vulnerability_analysis = province_vulnerability_analysis.withColumn(
    "poor_general_health_pct",
    spark_round((col("poor_general_health_count") / col("total_respondents")) * 100, 2)
)

province_vulnerability_analysis = province_vulnerability_analysis.withColumn(
    "poor_mental_health_pct",
    spark_round((col("poor_mental_health_count") / col("total_respondents")) * 100, 2)
)

province_vulnerability_analysis = province_vulnerability_analysis.withColumn(
    "no_regular_doctor_pct",
    spark_round((col("no_regular_doctor_count") / col("total_respondents")) * 100, 2)
)

# Round average chronic conditions
province_vulnerability_analysis = province_vulnerability_analysis.withColumn(
    "avg_chronic_conditions",
    spark_round(col("avg_chronic_conditions"), 2)
)

# Select final columns for display
result = province_vulnerability_analysis.select(
    "province",
    "province_label",
    "total_respondents",
    "high_vulnerability_count",
    "high_vulnerability_percentage",
    "weighted_high_vulnerability_percentage",
    "poor_general_health_pct",
    "poor_mental_health_pct",
    "no_regular_doctor_pct",
    "avg_chronic_conditions"
).orderBy(col("weighted_high_vulnerability_percentage").desc())

print("Vulnerability Analysis by Province")
print("=" * 80)
print()
display(result)

In [0]:
from pyspark.sql.functions import col, count, sum as spark_sum, avg, when, round as spark_round

# Group by has_regular_doctor and has_regular_doctor_label
doctor_vulnerability_analysis = analysis_df.groupBy("has_regular_doctor", "has_regular_doctor_label").agg(
    # Total respondents in each healthcare access group
    count("*").alias("total_respondents"),
    
    # Count of high vulnerability cases
    spark_sum(when(col("high_vulnerability_flag") == 1, 1).otherwise(0)).alias("high_vulnerability_count"),
    
    # Sum of survey weights for total and vulnerable populations
    spark_sum("survey_weight").alias("total_weighted_population"),
    spark_sum(when(col("high_vulnerability_flag") == 1, col("survey_weight")).otherwise(0)).alias("vulnerable_weighted_population"),
    
    # Count each vulnerability component
    spark_sum("poor_general_health_flag").alias("poor_general_health_count"),
    spark_sum("poor_mental_health_flag").alias("poor_mental_health_count"),
    spark_sum("has_difficulty_flag").alias("has_difficulty_count"),
    
    # Average chronic condition count
    avg("chronic_condition_count").alias("avg_chronic_conditions")
)

# Calculate high vulnerability percentage
doctor_vulnerability_analysis = doctor_vulnerability_analysis.withColumn(
    "high_vulnerability_percentage",
    spark_round((col("high_vulnerability_count") / col("total_respondents")) * 100, 2)
)

# Calculate weighted high vulnerability percentage
doctor_vulnerability_analysis = doctor_vulnerability_analysis.withColumn(
    "weighted_high_vulnerability_percentage",
    spark_round((col("vulnerable_weighted_population") / col("total_weighted_population")) * 100, 2)
)

# Calculate component percentages
doctor_vulnerability_analysis = doctor_vulnerability_analysis.withColumn(
    "poor_general_health_pct",
    spark_round((col("poor_general_health_count") / col("total_respondents")) * 100, 2)
)

doctor_vulnerability_analysis = doctor_vulnerability_analysis.withColumn(
    "poor_mental_health_pct",
    spark_round((col("poor_mental_health_count") / col("total_respondents")) * 100, 2)
)

doctor_vulnerability_analysis = doctor_vulnerability_analysis.withColumn(
    "difficulty_pct",
    spark_round((col("has_difficulty_count") / col("total_respondents")) * 100, 2)
)

# Round average chronic conditions
doctor_vulnerability_analysis = doctor_vulnerability_analysis.withColumn(
    "avg_chronic_conditions",
    spark_round(col("avg_chronic_conditions"), 2)
)

# Select final columns for display
result = doctor_vulnerability_analysis.select(
    "has_regular_doctor",
    "has_regular_doctor_label",
    "total_respondents",
    "high_vulnerability_count",
    "high_vulnerability_percentage",
    "weighted_high_vulnerability_percentage",
    "poor_general_health_pct",
    "poor_mental_health_pct",
    "difficulty_pct",
    "avg_chronic_conditions"
).orderBy(col("weighted_high_vulnerability_percentage").desc())

print("Vulnerability Analysis by Healthcare Access (Regular Doctor Status)")
print("=" * 80)
print()
display(result)